# German Declension Trainer — Level 1: Article + Nomen (All 4 Cases)

**How to use this notebook:**
1. Run **Cell 1** (setup & vocabulary database) once by pressing `Shift + Enter`.
2. Run **Cell 2** to start an interactive session (type `q` to quit anytime, or `stats` to see weak spots).
3. **Sentence Mode** (full sentences + English) and the **Plural Drill** are further down; both need Cell 1 run first.
4. Easily expand `VOCAB` in Cell 1 with nouns from your **Uni Bonn** course or **Instagram** saves!

In [1]:
import csv
import random
from collections import defaultdict

# Format: (gender, singular, plural, n_deklination_singular_ending, n_deklination_genitive_ending,
#          english_singular, english_plural)
# Note: Standard nouns have "" for n_deklination. Masculine weak nouns (n-Deklination) take -n/-en in Akk/Dat/Gen.
VOCAB = [
    ("m", "Tisch", "Tische", "", "", "table", "tables"),
    ("m", "Gedanke", "Gedanken", "n", "ns", "thought", "thoughts"),       # n-Deklination + Genitiv -ns
    ("m", "Student", "Studenten", "en", "en", "student", "students"),     # n-Deklination
    ("m", "Vorschlag", "Vorschläge", "", "", "suggestion", "suggestions"),
    ("m", "Alltag", "Alltage", "", "", "daily routine", "daily routines"),
    ("f", "Entscheidung", "Entscheidungen", "", "", "decision", "decisions"),
    ("f", "Erfahrung", "Erfahrungen", "", "", "experience", "experiences"),
    ("f", "Möglichkeit", "Möglichkeiten", "", "", "possibility", "possibilities"),
    ("f", "Woche", "Wochen", "", "", "week", "weeks"),
    ("f", "Sprache", "Sprachen", "", "", "language", "languages"),
    ("n", "Problem", "Probleme", "", "", "problem", "problems"),
    ("n", "Beispiel", "Beispiele", "", "", "example", "examples"),
    ("n", "Ergebnis", "Ergebnisse", "", "", "result", "results"),
    ("n", "Ziel", "Ziele", "", "", "goal", "goals"),
    ("n", "Jahr", "Jahre", "", "", "year", "years"),
]

# Genitiv Singular forms the rules below can't derive (masc./neuter, strong nouns).
# List every accepted form; the first one is shown as the "main" answer.
# e.g. "Herz": ["Herzens"], "Bus": ["Busses"], "Globus": ["Globus", "Globusses"]
GENITIV_OVERRIDES = {}

ARTICLES = {
    "bestimmt": {
        "Nominativ": {"m": "der", "f": "die", "n": "das", "pl": "die"},
        "Akkusativ": {"m": "den", "f": "die", "n": "das", "pl": "die"},
        "Dativ":     {"m": "dem", "f": "der", "n": "dem", "pl": "den"},
        "Genitiv":   {"m": "des", "f": "der", "n": "des", "pl": "der"},
    },
    "unbestimmt": {
        "Nominativ": {"m": "ein",   "f": "eine",  "n": "ein",   "pl": "keine"},
        "Akkusativ": {"m": "einen", "f": "eine",  "n": "ein",   "pl": "keine"},
        "Dativ":     {"m": "einem", "f": "einer", "n": "einem", "pl": "keinen"},
        "Genitiv":   {"m": "eines", "f": "einer", "n": "eines", "pl": "keiner"},
    },
}

# Possessive articles take the same endings as ein/kein (plural like keine/keinen/keiner)
POSSESSIVES = [("mein", "my"), ("dein", "your"), ("sein", "his"), ("ihr", "her"), ("ihr", "their"),
               ("unser", "our"), ("euer", "your"), ("Ihr", "your")]   # Ihr = formal "you"
POSSESSIVE_ENDINGS = {
    "Nominativ": {"m": "",   "f": "e",  "n": "",   "pl": "e"},
    "Akkusativ": {"m": "en", "f": "e",  "n": "",   "pl": "e"},
    "Dativ":     {"m": "em", "f": "er", "n": "em", "pl": "en"},
    "Genitiv":   {"m": "es", "f": "er", "n": "es", "pl": "er"},
}

def possessive(stem, case, slot):
    ending = POSSESSIVE_ENDINGS[case][slot]
    if stem == "euer" and ending:   # euer drops its e: eure, euren, eurem, eures
        stem = "eur"
    return stem + ending

# (German trigger, English translation)
CASE_TRIGGERS = {
    "Nominativ": [("Das Subjekt:", "the subject"), ("Dort steht/sind", "over there is/are"),
                  ("Das ist", "that is")],
    "Akkusativ": [("Ich sehe / brauche", "I see / need"), ("Ohne", "without"), ("Für", "for"),
                  ("Es geht um", "it's about")],
    "Dativ":     [("Mit", "with"), ("Nach", "after"), ("Aus", "out of / from"),
                  ("Bei", "at / at someone's place"), ("Gegenüber", "opposite")],
    "Genitiv":   [("Trotz", "despite"), ("Wegen", "because of"), ("Während", "during"),
                  ("Innerhalb", "within"), ("Aufgrund", "due to")],
}

# Drill 1 sentences live in drill1_corpus.tsv: one real sentence per row, with its own article and number.
# article = bestimmt | unbestimmt | kein | mein | dein | sein | ihr | unser | euer | Ihr ; number = sg | pl
# The answer column is checked against the declension rules when Cell 1 runs.
# VOCAB nouns without any sentence in the file are still drilled, with a random trigger and no example.

# Suffixes that only take -s in Genitiv (des Museums, des Mädchens, des Königs, des Teppichs, des Autos)
S_ONLY_ENDINGS = ("um", "chen", "lein", "ling", "ig", "ich", "a", "e", "i", "o", "u", "y")
# Unstressed -el/-en/-er/-em after a consonant also only take -s (des Onkels, des Lehrers, des Problems),
# but NOT after a vowel, where the syllable is stressed (des Ziel(e)s, des Beispiel(e)s, des Bier(e)s)
UNSTRESSED_ENDINGS = ("el", "en", "er", "em")
# s-sounds always take -es (des Hauses, des Fußes, des Reflexes, des Satzes)
ES_ONLY_ENDINGS = ("s", "ß", "x", "z")

def genitive_forms(singular):
    """Accepted Genitiv Singular forms of a strong masculine/neuter noun."""
    if singular in GENITIV_OVERRIDES:
        return list(GENITIV_OVERRIDES[singular])
    low = singular.lower()
    if low.endswith("nis"):                 # Ergebnis -> Ergebnisses (s doubles)
        return [singular + "ses"]
    if low.endswith(ES_ONLY_ENDINGS):
        return [singular + "es"]
    if low.endswith(S_ONLY_ENDINGS):
        return [singular + "s"]
    if low.endswith(UNSTRESSED_ENDINGS) and len(low) > 2 and low[-3] not in "aeiouäöü":
        return [singular + "s"]
    # Everything else: both are correct (des Tisch(e)s, des Jahr(e)s, des Ziel(e)s, des Vorschlag(e)s)
    return [singular + "es", singular + "s"]

def inflect_noun(gender, singular, plural, n_end, n_gen, case, is_plural):
    """Return the list of accepted noun forms (first = main answer)."""
    if is_plural:
        # Dativ Plural always adds -n unless plural already ends in -n or -s
        if case == "Dativ" and not plural.endswith(("n", "s")):
            return [plural + "n"]
        return [plural]

    # Masculine n-Deklination (e.g., der Student -> den Studenten)
    if gender == "m" and n_end:
        if case == "Genitiv":
            return [singular + (n_gen if n_gen else n_end)]
        if case in ("Akkusativ", "Dativ"):
            return [singular + n_end]
        return [singular]

    # Standard Genitiv Singular for masculine & neuter (-s or -es)
    if case == "Genitiv" and gender in ("m", "n"):
        return genitive_forms(singular)

    return [singular]

TRIGGER_CASE = {t: case for case, items in CASE_TRIGGERS.items() for t, _ in items}
TRIGGER_EN = {t: en for items in CASE_TRIGGERS.values() for t, en in items}
# Plural form of triggers that contain a verb
TRIGGER_PLURAL = {"Das ist": ("Das sind", "those are"), "Dort steht/sind": ("Dort stehen/sind", "over there are")}
VOCAB_BY_NOUN = {v[1]: v for v in VOCAB}
POSSESSIVE_EN = {"mein": "my", "dein": "your", "sein": "his", "ihr": "her/their",
                 "unser": "our", "euer": "your", "Ihr": "your", "kein": "no"}

def article_for(art, case, slot):
    if art in ("bestimmt", "unbestimmt"):
        return ARTICLES[art][case][slot]
    return possessive(art, case, slot)          # kein- and possessives share the ein-endings

def answers_for(noun, case, art, is_plural):
    gender, sg, pl, n_end, n_gen, _, _ = VOCAB_BY_NOUN[noun]
    slot = "pl" if is_plural else gender
    article = article_for(art, case, slot)
    return [f"{article} {form}" for form in inflect_noun(gender, sg, pl, n_end, n_gen, case, is_plural)]

DRILL1_CORPUS = []
with open("drill1_corpus.tsv", encoding="utf-8") as f:
    for row in csv.DictReader(f, delimiter="\t"):
        where = f"drill1_corpus.tsv: {row['german']}"
        assert row["noun"] in VOCAB_BY_NOUN, f"Noun not in VOCAB ({where})"
        assert row["trigger"] in TRIGGER_CASE, f"Unknown trigger {row['trigger']!r} ({where})"
        assert row["case"] == TRIGGER_CASE[row["trigger"]], f"Wrong case for {row['trigger']!r} ({where})"
        assert row["article"] in ("bestimmt", "unbestimmt") or row["article"] in POSSESSIVE_EN, f"Unknown article ({where})"
        assert not (row["article"] == "unbestimmt" and row["number"] == "pl"), f"ein- has no plural, use kein ({where})"
        expected = answers_for(row["noun"], row["case"], row["article"], row["number"] == "pl")
        assert row["answer"] in expected, f"Answer {row['answer']!r} should be {' / '.join(expected)} ({where})"
        assert row["german"].count("___") == 1, f"Need exactly one ___ ({where})"
        DRILL1_CORPUS.append(row)

def article_hint(art, is_plural, en_sg, en_pl):
    """Tag label and English noun phrase, e.g. ("possessiv (mein-)", "my table")."""
    noun_en = en_pl if is_plural else en_sg
    if art == "bestimmt":
        return "bestimmt", f"the {noun_en}"
    if art == "unbestimmt":
        return "unbestimmt", f"{'an' if en_sg[0] in 'aeiou' else 'a'} {en_sg}"
    if art == "kein":
        return "negativ (kein-)", f"no {noun_en}"
    return f"possessiv ({art}-{', formell' if art == 'Ihr' else ''})", f"{POSSESSIVE_EN[art]} {noun_en}"

def generate_item(filter_case=None):
    rows = [r for r in DRILL1_CORPUS if filter_case in (None, r["case"])]
    nouns_in_corpus = {r["noun"] for r in DRILL1_CORPUS}
    # every noun with a fitting sentence, plus VOCAB nouns that have no sentences at all yet
    candidates = sorted({r["noun"] for r in rows} | (VOCAB_BY_NOUN.keys() - nouns_in_corpus))
    noun = random.choice(candidates)
    gender, sg, pl, n_end, n_gen, en_sg, en_pl = VOCAB_BY_NOUN[noun]

    noun_rows = [r for r in rows if r["noun"] == noun]
    if noun_rows:
        row = random.choice(noun_rows)
        case, trigger, art, is_plural = row["case"], row["trigger"], row["article"], row["number"] == "pl"
    else:  # noun without sentences: random trigger and article, no example
        row = None
        is_plural = random.choice([False, False, True])
        case = filter_case if filter_case else random.choice(list(ARTICLES["bestimmt"].keys()))
        trigger, _ = random.choice(CASE_TRIGGERS[case])
        art = random.choice(["bestimmt", "kein" if is_plural else "unbestimmt"])

    slot = "pl" if is_plural else gender
    answers = answers_for(noun, case, art, is_plural)
    if row:  # show the form written in the corpus first
        answers = [row["answer"]] + [a for a in answers if a != row["answer"]]

    trigger_en = TRIGGER_EN[trigger]
    if is_plural:
        trigger, trigger_en = TRIGGER_PLURAL.get(trigger, (trigger, trigger_en))
    art_label, en_np = article_hint(art, is_plural, en_sg, en_pl)
    num_label = f"Plural (Sg: {sg})" if is_plural else f"Singular ({'der' if gender=='m' else 'die' if gender=='f' else 'das'} {sg})"
    prompt = f"[{case} | {art_label} | {num_label}]  →  {trigger} ___   ({en_np}, {trigger_en})"

    # (sentence with gap, full sentence, English)
    example = None
    if row:
        full = row["german"].replace("___", answers[0])
        example = (row["german"], full[:1].upper() + full[1:], row["english"])
    return prompt, answers, case, slot, example

def normalize(text):
    """Collapse extra/leading/trailing whitespace."""
    return " ".join(text.split())

def check_answer(user_ans, answers):
    """Return 'exact', 'case' (right except capitalisation) or None."""
    ans = normalize(user_ans)
    # The article may be capitalised (sentence start); only the noun's capitalisation matters.
    art, _, noun = ans.partition(" ")
    if any(art.casefold() == a_art.casefold() and noun == a_noun
           for a_art, _, a_noun in (a.partition(" ") for a in answers)):
        return "exact"
    if ans.casefold() in (a.casefold() for a in answers):
        return "case"
    return None

SLOT_LABELS = {"m": "maskulin", "f": "feminin", "n": "neutral", "pl": "Plural"}

def new_stats():
    # key -> [correct, total]
    return {"case": defaultdict(lambda: [0, 0]), "slot": defaultdict(lambda: [0, 0])}

def record(stats, case, slot, correct):
    for key, value in (("case", case), ("slot", slot)):
        stats[key][value][1] += 1
        stats[key][value][0] += int(correct)

def print_stats(stats):
    if not stats["case"]:
        print("No answers yet.")
        return
    for key, title in (("case", "By case"), ("slot", "By gender / number")):
        print(f"\n{title} (weakest first):")
        rows = sorted(stats[key].items(), key=lambda kv: kv[1][0] / kv[1][1])
        for value, (right, n) in rows:
            label = SLOT_LABELS.get(value, value)
            print(f"  {label:<10} {right:>3}/{n:<3} ({round(right / n * 100)}%)")

print("Setup complete! Run the next cell to start drilling.")

Setup complete! Run the next cell to start drilling.


In [1]:
# Interactive Drill Loop
# Optional: set FOCUS_CASE = "Genitiv" (or "Nominativ", "Akkusativ", "Dativ") to drill one case only, or None for all 4.
FOCUS_CASE = None

score = 0
total = 0
mistakes = []
stats = new_stats()

print("Type 'q' to finish and see your summary, 'stats' to see your weak spots.\n" + "-" * 55)

prompt, answers, case, slot, example = generate_item(FOCUS_CASE)
while True:
    print(f"\n{prompt}")
    if example:
        print(f"   z.B. {example[0]}  ({example[2]})")
    user_ans = input("Your answer: ").strip()

    if user_ans.lower() in ("q", "quit", "exit"):
        break
    if user_ans.lower() == "stats":
        print_stats(stats)
        continue  # same question again, not counted

    total += 1
    result = check_answer(user_ans, answers)
    record(stats, case, slot, result is not None)
    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})")
        if result == "case":
            print(f"   ⚠️ Achte auf die Großschreibung: '{answers[0]}'")
        if len(answers) > 1:
            print(f"   (Beide Formen sind korrekt: {' / '.join(answers)})")
    else:
        correct = " / ".join(answers)
        mistakes.append((prompt, user_ans, correct))
        print(f"❌ Falsch! Dein Input: '{user_ans}'  →  Richtig: '{correct}'")
    if example:
        print(f"   → {example[1]}")

    prompt, answers, case, slot, example = generate_item(FOCUS_CASE)

print("\n" + "=" * 55)
if total > 0:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for p, u, e in mistakes:
            print(f"  • {p}\n    You wrote: {u}  |  Correct: {e}")
else:
    print("Session ended.")

NameError: name 'new_stats' is not defined

## Sentence Mode: Real sentences + case triggers

Loads `sentence_corpus.tsv` (180 sentences with English translations). Fill the gap with **article + noun**.
Each prompt shows the case trigger (verb, preposition, wo?/wohin?, Genitivattribut, …) and the noun in its dictionary form.
Edit the TSV in any spreadsheet app to add your own sentences. Separate alternative answers with `|`.

In [ ]:
# Sentence Mode Drill (needs Cell 1 to be run first)
import csv

FOCUS_CASE = None          # e.g. "Dativ", or None for all cases
FOCUS_TRIGGER_TYPE = None  # e.g. "Wechselpräposition" (substring match), or None
SHOW_ENGLISH_FIRST = False # True = show the translation before you answer

with open("sentence_corpus.tsv", encoding="utf-8") as f:
    CORPUS = list(csv.DictReader(f, delimiter="\t"))

pool = [r for r in CORPUS
        if (FOCUS_CASE is None or r["case"] == FOCUS_CASE)
        and (FOCUS_TRIGGER_TYPE is None or FOCUS_TRIGGER_TYPE.lower() in r["trigger_type"].lower())]
print(f"{len(pool)} sentences loaded. Type 'q' to quit, 'stats' for weak spots.\n" + "-" * 55)

GENDER_OF = {"der": "m", "die": "f", "das": "n"}
score = total = 0
mistakes = []
stats = new_stats()
random.shuffle(pool)

for row in pool:
    answers = [a.strip() for a in row["answer"].split("|")]
    slot = "pl" if row["number"] == "pl" else GENDER_OF[row["noun"].split()[0]]
    num = " (Pl.)" if row["number"] == "pl" else ""
    print(f"\n[{row['trigger']} · {row['trigger_type']}]   ({row['noun']}{num}, {row['article']})")
    print(f"  {row['german']}")
    if SHOW_ENGLISH_FIRST:
        print(f"  {row['english']}")

    while True:
        user_ans = input("Your answer: ").strip()
        if user_ans.lower() == "stats":
            print_stats(stats)
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    result = check_answer(user_ans, answers)
    record(stats, row["case"], slot, result is not None)
    full = row["german"].replace("___", answers[0])
    if result is not None:
        score += 1
        print(f"✅ Richtig! ({score}/{total})  →  {full}")
        if result == "case":
            print(f"   ⚠️ Achte auf die Großschreibung: '{answers[0]}'")
        if len(answers) > 1:
            print(f"   (Beide Formen sind korrekt: {' / '.join(answers)})")
    else:
        mistakes.append((row["german"], user_ans, " / ".join(answers), row["case"], row["trigger"]))
        print(f"❌ Falsch! ({row['case']})  →  {full}")
    if not SHOW_ENGLISH_FIRST:
        print(f"   {row['english']}")
else:
    print("\n🎉 You went through the whole corpus!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_stats(stats)
    if mistakes:
        print("\nReview these slips:")
        for s, u, c, case, trig in mistakes:
            print(f"  • {s}  [{case} ← {trig}]\n    You wrote: {u}  |  Correct: {c}")
else:
    print("Session ended.")

## Plural Drill: Learn the plural patterns

Loads `plural_nouns.tsv` (149 nouns, each tagged with its plural pattern). You see `der Stuhl (chair)` → type the plural (`Stühle`, with or without `die`).
On a mistake it tells you **which rule applies** and spots a missing or wrong umlaut. Type `ae/oe/ue` if you have no umlaut keys.
Add your own nouns to the TSV; the `pattern` column must be one of the keys in `PLURAL_RULES` below.

In [ ]:
# Plural Drill
import csv

FOCUS_GENDER = None    # "der", "die", "das", or None for all
FOCUS_PATTERN = None   # a key from PLURAL_RULES (e.g. "m_uml_e"), or None

PLURAL_RULES = {
    "f_en":       "Feminin → -(e)n. -ung, -heit, -keit, -schaft, -ion, -tät, -ik, -ei always take -en (~90% of feminine nouns).",
    "f_n":        "Feminin on -e, -el, -er → just add -n.",
    "f_nen":      "Feminin on -in → -innen (double n!).",
    "f_uml_e":    "Exception: short feminine nouns (Stadt, Hand, Nacht, Wand, Wurst, Maus) → ¨-e.",
    "f_uml":      "Exception: die Mutter, die Tochter → umlaut only, no ending.",
    "nis":        "-nis → -nisse (the s doubles).",
    "m_weak":     "n-Deklination (masc. -e people/animals, -ent, -ant, -ist, -oge, + Mensch, Nachbar, Herr, Name, Gedanke) → -(e)n.",
    "m_e":        "Masculine → -e, here WITHOUT umlaut. Memorise it: der Tag, -e.",
    "m_uml_e":    "Masculine → -e WITH umlaut. Memorise it: der Stuhl, ¨-e.",
    "m_zero":     "Masculine on -el, -en, -er → no ending, no umlaut.",
    "m_uml_zero": "Masculine on -el, -en, -er → no ending, but WITH umlaut.",
    "m_uml_er":   "Small masculine group → ¨-er (Mann, Wald, Mund, Gott).",
    "n_er":       "Short neuter nouns → -er, with umlaut wherever possible (a/o/u/au).",
    "n_e":        "Neuter → -e, and neuter -e plurals NEVER take an umlaut.",
    "n_zero":     "Neuter on -chen, -lein, -el, -er, -en or Ge-…-e → no change.",
    "n_en":       "Small neuter group → -(e)n (Auge, Ende, Ohr, Bett, Hemd, Herz).",
    "um":         "-um → -en (Museen, Zentren) or -a (Praktika).",
    "s":          "Foreign words and nouns ending in a vowel (-o, -a, -y, -é) → -s.",
}

with open("plural_nouns.tsv", encoding="utf-8") as f:
    NOUNS = list(csv.DictReader(f, delimiter="\t"))
unknown = {r["pattern"] for r in NOUNS} - PLURAL_RULES.keys()
assert not unknown, f"Unknown pattern(s) in plural_nouns.tsv: {unknown}"

UMLAUT_STRIP = str.maketrans("äöüÄÖÜ", "aouAOU")
def translit(s):
    for a, b in (("ä", "ae"), ("ö", "oe"), ("ü", "ue"), ("Ä", "Ae"), ("Ö", "Oe"), ("Ü", "Ue")):
        s = s.replace(a, b)
    return s

def check_plural(user, correct):
    """Return (ok, note)."""
    ans = " ".join(user.split())
    if ans.lower().startswith("die "):
        ans = ans[4:]
    if ans == correct:
        return True, None
    if ans == translit(correct):
        return True, f"Mit Umlaut geschrieben: {correct}"
    if ans.casefold() in (correct.casefold(), translit(correct).casefold()):
        return True, f"Achte auf die Großschreibung: {correct}"
    if ans.translate(UMLAUT_STRIP) == correct.translate(UMLAUT_STRIP) or translit(ans) == translit(correct):
        if correct.translate(UMLAUT_STRIP) == correct:
            return False, "No umlaut here!"
        return False, "Umlaut missing!" if ans.translate(UMLAUT_STRIP) == ans else "Umlaut on the wrong vowel!"
    return False, None

pool = [r for r in NOUNS
        if (FOCUS_GENDER is None or r["article"] == FOCUS_GENDER)
        and (FOCUS_PATTERN is None or r["pattern"] == FOCUS_PATTERN)]
random.shuffle(pool)
print(f"{len(pool)} nouns loaded. Type 'q' to quit, 'stats' for weak spots.\n" + "-" * 55)

pstats = {"pattern": defaultdict(lambda: [0, 0]), "article": defaultdict(lambda: [0, 0])}
def print_plural_stats():
    if not pstats["pattern"]:
        print("No answers yet.")
        return
    for key, title in (("article", "By gender"), ("pattern", "By plural pattern")):
        print(f"\n{title} (weakest first):")
        for value, (right, n) in sorted(pstats[key].items(), key=lambda kv: kv[1][0] / kv[1][1]):
            print(f"  {value:<11} {right:>3}/{n:<3} ({round(right / n * 100)}%)")

score = total = 0
mistakes = []
for row in pool:
    print(f"\n{row['article']} {row['singular']} ({row['english']})  →  die ___")
    while True:
        user_ans = input("Plural: ").strip()
        if user_ans.lower() == "stats":
            print_plural_stats()
            continue
        break
    if user_ans.lower() in ("q", "quit", "exit"):
        break

    total += 1
    ok, note = check_plural(user_ans, row["plural"])
    for key in ("pattern", "article"):
        pstats[key][row[key]][1] += 1
        pstats[key][row[key]][0] += int(ok)
    if ok:
        score += 1
        print(f"✅ Richtig! die {row['plural']} ({score}/{total})")
        if note:
            print(f"   ⚠️ {note}")
    else:
        mistakes.append((row, user_ans))
        print(f"❌ Falsch!{' ' + note if note else ''}  →  die {row['plural']}")
        print(f"   Regel: {PLURAL_RULES[row['pattern']]}")
else:
    print("\n🎉 You went through all the nouns!")

print("\n" + "=" * 55)
if total:
    print(f"Session finished: {score}/{total} correct ({round(score/total*100)}%)")
    print_plural_stats()
    if mistakes:
        print("\nReview these slips:")
        for row, u in mistakes:
            print(f"  • {row['article']} {row['singular']} → die {row['plural']}   (you wrote: {u})")
else:
    print("Session ended.")